In [11]:
import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchtyping import TensorType
from typing import List
import numpy as np
import math

In [12]:
# Normalization for the training dataset
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), # flop image 50% of the time
    transforms.RandomCrop(32, padding=4), # shift image slightly
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Normalization for the test dataset
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Using the CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Loading the 10,000 image official test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# Setting the seed
torch.manual_seed(0)

In [13]:
# Vision  Transformers class
class VisionTransformer(nn.Module):
  def __init__(self, patch_size, num_heads, num_blocks, img_size, model_dim, num_classes):
    super().__init__()
    self.num_patches = (img_size//patch_size)**2
    self.transformer_blocks = nn.ModuleList()
    for _ in range(num_blocks):
        self.transformer_blocks.append(self.TransformerBlock(model_dim,num_heads))

    self.patch_embeddings = nn.Conv2d(in_channels = 3, out_channels=model_dim, kernel_size=patch_size, stride=patch_size)
    self.embedding_dropout = nn.Dropout(p=0.1)
    self.position_embeddings = nn.Parameter(torch.zeros(1, self.num_patches+1, model_dim))
    self.cls_token = nn.Parameter(torch.zeros(1,1,model_dim))
    self.final_norm = nn.LayerNorm(model_dim)
    self.classifier = nn.Linear(model_dim, num_classes)

  def forward(self, images: TensorType[float]):
    patch_embeddings = self.patch_embeddings(images)
    patch_embeddings = patch_embeddings.flatten(2)
    patch_embeddings = patch_embeddings.transpose(1,2)
    cls_token = self.cls_token.expand(images.shape[0],-1,-1)
    x = torch.cat((cls_token, patch_embeddings), dim=1)
    x = self.embedding_dropout(x + self.position_embeddings)
    for block in self.transformer_blocks:
        x = block(x)
    x = self.final_norm(x)
    output = self.classifier(x[:, 0])
    return output


  class TransformerBlock(nn.Module):
      def __init__(self, model_dim: int, num_heads: int):
        super().__init__()

        self.attention = self.MultiHeadSelfAttention(model_dim, num_heads)
        self.linear_network = self.NeuralNetwork(model_dim)
        self.first_norm = nn.LayerNorm(model_dim)
        self.second_norm = nn.LayerNorm(model_dim)


      def forward(self, embedded: TensorType[float]):
        embedded = embedded + self.attention(self.first_norm(embedded))
        embedded = embedded + self.linear_network(self.second_norm(embedded))
        return embedded

      class MultiHeadSelfAttention(nn.Module):
        def __init__(self, model_dim: int, num_heads: int):
          super().__init__()
          self.num_heads = num_heads
          self.head_size = model_dim // self.num_heads
          self.q = nn.Linear(model_dim, model_dim, bias=False)
          self.v = nn.Linear(model_dim, model_dim, bias=False)
          self.k = nn.Linear(model_dim, model_dim, bias=False)
          self.dropout = nn.Dropout(p=0.1)
          self.out_projection = nn.Linear(model_dim,model_dim)

        def forward(self, embedded: TensorType[float]):
          B, T, C = embedded.shape
          q = self.q(embedded)
          q = q.view(B, T, self.num_heads, self.head_size).transpose(1,2)

          k = self.k(embedded)
          k = k.view(B, T, self.num_heads, self.head_size).transpose(1,2)

          v = self.v(embedded)
          v = v.view(B, T, self.num_heads, self.head_size).transpose(1,2)

          scores = q @ k.transpose(-2,-1)
          scores = scores/(self.head_size**0.5)
          scores = nn.functional.softmax(scores, dim=2)
          output = self.dropout(scores) @ v
          output = output.transpose(1,2).contiguous().view(B, T, self.head_size*self.num_heads)
          output = self.out_projection(output)
          return output

      class NeuralNetwork(nn.Module):
          def __init__(self, model_dim: int):
            super().__init__()

            self.layer1 = nn.Linear(model_dim, model_dim*4)
            self.gelu = nn.GELU()
            self.dropout_1 = nn.Dropout(p=0.2)
            self.layer2 = nn.Linear(model_dim*4, model_dim)

          def forward(self, embedded: TensorType[float]):
            x = self.layer1(embedded)
            x = self.gelu(x)
            x = self.dropout_1(x)
            x = self.layer2(x)

            return x


In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = VisionTransformer(
    patch_size=4,
    num_heads=4,
    num_blocks=6,
    img_size=32,
    model_dim=128,
    num_classes=10
).to(device)

In [15]:
# Training loop
def train(model, trainloader, learning_rate, epochs):
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)
  total_steps = len(trainloader) * epochs
  warmup_steps = total_steps // 10

  def lr_lambda(step):
      if step < warmup_steps:
          return (step + 1) / warmup_steps
      progress = (step - warmup_steps) / (total_steps - warmup_steps)
      return 0.5 * (1 + math.cos(math.pi * progress))

  scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
  step_count = 0

  for epoch in range(epochs):
      running_loss = 0.0
      for i, (images, labels) in enumerate(trainloader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        step_count += 1
        running_loss += loss.item()
        if i % 100 == 0:
          print(f'Epoch {epoch+1}, Step {i}, Loss: {loss.item():.3f}, Avg: {running_loss/(i+1):.3f}, LR: {scheduler.get_last_lr()[0]:.6f}')

        if (epoch + 1) % 10 == 0:
          torch.save(model.state_dict(), f'vit_epoch{epoch+1}.pth')

In [16]:
# Evaluation loop
def validate(model, testloader):
  model.eval()
  correct, total = 0, 0
  with torch.no_grad():
    for images, labels in testloader:
      images, labels = images.to(device), labels.to(device)
      output = model(images)
      predicted = output.argmax(dim=1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
  print(f'Accuracy: {100 * correct / total:.2f}%')
  model.train()


In [17]:
def overfit_test(model, trainloader, device):
    images, labels = next(iter(trainloader))
    images, labels = images.to(device), labels.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for step in range(200):
        optimizer.zero_grad()
        outputs = model(images)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        if step % 20 == 0:
            print(f'Step {step}, Loss: {loss.item():.3f}')

overfit_test(model, trainloader, device)

Step 0, Loss: 2.352
Step 20, Loss: 0.515
Step 40, Loss: 0.042
Step 60, Loss: 0.009
Step 80, Loss: 0.005
Step 100, Loss: 0.004
Step 120, Loss: 0.003
Step 140, Loss: 0.003
Step 160, Loss: 0.002
Step 180, Loss: 0.002


In [18]:
# Reinitialize fresh model for real training
model = VisionTransformer(
    patch_size=4, num_heads=4, num_blocks=6,
    img_size=32, model_dim=128, num_classes=10
).to(device)


# training below, unchanged
train(model, trainloader, epochs=50, learning_rate=3e-4)
validate(model, testloader)

Epoch 1, Step 0, Loss: 2.377, Avg: 2.377, LR: 0.000000
Epoch 1, Step 100, Loss: 2.120, Avg: 2.335, LR: 0.000008
Epoch 1, Step 200, Loss: 1.922, Avg: 2.226, LR: 0.000015
Epoch 1, Step 300, Loss: 2.226, Avg: 2.158, LR: 0.000023
Epoch 1, Step 400, Loss: 2.100, Avg: 2.118, LR: 0.000031
Epoch 1, Step 500, Loss: 1.950, Avg: 2.086, LR: 0.000039
Epoch 1, Step 600, Loss: 1.897, Avg: 2.059, LR: 0.000046
Epoch 1, Step 700, Loss: 1.847, Avg: 2.033, LR: 0.000054
Epoch 2, Step 0, Loss: 1.999, Avg: 1.999, LR: 0.000060
Epoch 2, Step 100, Loss: 1.854, Avg: 1.823, LR: 0.000068
Epoch 2, Step 200, Loss: 1.757, Avg: 1.815, LR: 0.000075
Epoch 2, Step 300, Loss: 1.774, Avg: 1.806, LR: 0.000083
Epoch 2, Step 400, Loss: 1.658, Avg: 1.802, LR: 0.000091
Epoch 2, Step 500, Loss: 1.707, Avg: 1.791, LR: 0.000099
Epoch 2, Step 600, Loss: 1.790, Avg: 1.776, LR: 0.000106
Epoch 2, Step 700, Loss: 1.642, Avg: 1.765, LR: 0.000114
Epoch 3, Step 0, Loss: 1.709, Avg: 1.709, LR: 0.000120
Epoch 3, Step 100, Loss: 1.751, Avg: 